# Imports

In [20]:
# Import modules
import tensorflow as tf

import numpy as np

from keras.applications.vgg16 import VGG16
from keras.layers import Dropout
from keras.layers import Flatten, Dense
from keras.models import Model, save_model
from keras.optimizers import Adam

from tqdm import tqdm

In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent.parent))

from preprocessing.audio_processor import preprocess
from config import AUDIO_DIR, AUDIO_CONFIG

🖥️  Using device: cuda


# Data Preprocessing

In [3]:
# Load data parameters
tf.random.set_seed(42)

IMG_HEIGHT = AUDIO_CONFIG["spectrogram_height"]
IMG_WIDTH = AUDIO_CONFIG["spectrogram_width"]
CHANNELS = 3

INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

EPOCHS = 5
BATCH_SIZE = 32

In [5]:
audio_data = {
    'training': {
        'spectrograms': [],
        'labels': []
    },
    'testing': {
        'spectrograms': [],
        'labels': []
    }
}

# Track skipped files
skipped_files = []

for split in ['training', 'testing']:
    for label in ['real', 'fake']:
        audio_files = list((AUDIO_DIR / split / label).glob('*.wav')) + list((AUDIO_DIR / split / label).glob('*.mp3'))
        print(f"\nProcessing {label} audio files in {split} set:")
        
        for audio in tqdm(audio_files, desc=f"{split}/{label}"):
            try:
                spectrogram, _ = preprocess(audio)
                
                audio_data[split]['spectrograms'].append(spectrogram)
                audio_data[split]['labels'].append(0 if label=='real' else 1)
            except Exception as e:
                # Skip corrupted or unreadable files
                skipped_files.append((str(audio), str(e)))
                continue

# Report skipped files
if skipped_files:
    print(f"\nSkipped {len(skipped_files)} corrupted/unreadable files:")
    for file_path, error in skipped_files:
        print(f"  - {file_path}")
else:
    print(f"\nAll files processed successfully!")



Processing real audio files in training set:


training/real: 100%|██████████| 4000/4000 [00:49<00:00, 80.78it/s] 



Processing fake audio files in training set:


training/fake: 100%|██████████| 4000/4000 [00:38<00:00, 103.46it/s]



Processing real audio files in testing set:


testing/real: 100%|██████████| 1000/1000 [00:23<00:00, 42.01it/s]



Processing fake audio files in testing set:


testing/fake: 100%|██████████| 1000/1000 [00:19<00:00, 51.00it/s]


Skipped 1 corrupted/unreadable files:
  - d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\!S9\Explainability AI\Project\XAI_Final_Project\data\audio\training\fake\file13424.mp3


In [6]:
# Convert lists to NumPy arrays and create TensorFlow datasets
train_tensors = np.array(audio_data['training']['spectrograms'])
train_labels = np.array(audio_data['training']['labels'])

test_tensors = np.array(audio_data['testing']['spectrograms'])
test_labels = np.array(audio_data['testing']['labels'])

print(f"Train shape: {train_tensors.shape}, Labels: {train_labels.shape}")
print(f"Test shape: {test_tensors.shape}, Labels: {test_labels.shape}")

train_ds = tf.data.Dataset.from_tensor_slices((train_tensors, train_labels)).shuffle(1000).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((test_tensors, test_labels)).shuffle(100).batch(BATCH_SIZE)

print(f"\nTrain batches: {len(train_ds)}")
print(f"Test batches: {len(test_ds)}")

Train shape: (7999, 224, 224, 3), Labels: (7999,)
Test shape: (2000, 224, 224, 3), Labels: (2000,)

Train batches: 250
Test batches: 63


In [ ]:
import gc

del audio_data, train_tensors, train_labels, test_tensors, test_labels

gc.collet()

# Model

In [8]:
vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)

In [9]:
for layer in vgg16.layers:
    layer.trainable = False

In [10]:
x = Flatten()(vgg16.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(1, activation='sigmoid')(x)

In [11]:
model_vgg = Model(inputs=vgg16.input, outputs=x)

In [12]:
model_vgg.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=1e-5), metrics=['accuracy'])

In [13]:
model_vgg.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,137,729 (80.63 MB)

 Trainable params: 6,423,041 (24.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [14]:
history = model_vgg.fit(
    train_ds,
    validation_data=test_ds,
    shuffle=True,
    epochs=EPOCHS
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 710s 3s/step - accuracy: 0.9315 - loss: 0.3523 - val_accuracy: 0.6875 - val_loss: 1.3797
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 676s 3s/step - accuracy: 0.9464 - loss: 0.2359 - val_accuracy: 0.8265 - val_loss: 0.7394
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 684s 3s/step - accuracy: 0.9642 - loss: 0.1173 - val_accuracy: 0.8255 - val_loss: 0.6658
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 670s 3s/step - accuracy: 0.9716 - loss: 0.0914 - val_accuracy: 0.8895 - val_loss: 0.3668
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 828s 3s/step - accuracy: 0.9760 - loss: 0.0619 - val_accuracy: 0.8945 - val_loss: 0.3246


In [22]:
save_model(model_vgg, filepath="./vgg16.h5")